# TriadLM #6 — 125M-class base on corpus_v2 (Kaggle free T4)

The last planned model. 12L/768, block 512 (fits 16GB), 1500 steps ~45 min.
Needs: **GPU T4 ON**, **Internet ON**, corpus_v2 on the Hub (#2 done).
Checkpoints every 500 steps — interruptions are free to resume.

In [ ]:
!pip install -q torch tokenizers pyyaml tqdm "pydantic>=2" requests huggingface_hub
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-ONLY — enable GPU!")
import os
def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ[name]
print("ready")

In [ ]:
!git clone https://github.com/PhillipMtalika/triadlm.git
%cd triadlm
!pwd && ls
!python -m pytest tests/test_tokenizer.py tests/test_model.py -q 2>&1 | tail -n 1

In [ ]:
# Pull corpus_v2 shards + manifest + tokenizer from the Hub (no rebuild).
from huggingface_hub import snapshot_download, HfApi
token = get_secret("HF_TOKEN")
me = HfApi(token=token).whoami(token)["name"]
import shutil, os, glob as _g
dst = snapshot_download(repo_id=f"{me}/triadlm-corpus-v2", repo_type="dataset", token=token)
os.makedirs("data/shards/corpus_v2", exist_ok=True)
os.makedirs("data/manifests", exist_ok=True)
os.makedirs("data/checkpoints/base_50m/tokenizer", exist_ok=True)
n = 0
for f in _g.glob(os.path.join(dst, "*.pt")):
    shutil.copy(f, "data/shards/corpus_v2/")
    n += 1
shutil.copy(os.path.join(dst, "corpus_v2.json"), "data/manifests/corpus_v2.json")
for f in _g.glob(os.path.join(dst, "tokenizer", "*")):
    shutil.copy(f, "data/checkpoints/base_50m/tokenizer/")
print(f"shards: {n}")

In [ ]:
# Train (~45 min). If interrupted: resume from the newest step_*.pt.
!python -m triadlm.train --config configs/kaggle_125m.yaml
# !ls data/checkpoints/kaggle_125m/
# !python -m triadlm.train --config configs/kaggle_125m.yaml --resume data/checkpoints/kaggle_125m/step_1000.pt

In [ ]:
!python -m evals.run_eval --checkpoint data/checkpoints/kaggle_125m/final.pt --variant base --out experiments/runs/kaggle-125m.json 2>&1 | grep -E '"(run_id|factuality|instruction_following|over_refusal_rate|chichewa_gap|val_loss|perplexity)"'
!head -n 2 experiments/registry.csv

In [ ]:
from huggingface_hub import HfApi, create_repo
token = get_secret("HF_TOKEN")
api = HfApi(token=token)
me = api.whoami(token)["name"]
create_repo(f"{me}/triadlm-m1-125m", private=False, exist_ok=True, token=token)
api.upload_folder(folder_path="data/checkpoints/kaggle_125m", repo_id=f"{me}/triadlm-m1-125m", token=token)
print("uploaded 125m")